# Capítulo 2 — Fundamentos de flujo a través de tuberías

Ecuación general de energía, pérdidas por fricción (Darcy, Fanning) y factor de fricción.

> **Fuente.** Métodos y datos de *Transporte de Hidrocarburos por Ductos*, F. Garaicochea P.,
> C. Bernal H. y O. López O. (CIPM, 1991; reed. 2022), verificados contra la tesis UNAM de 1990 que
> le dio origen (Bernal y López, asesor Garaicochea). Los enunciados están **parafraseados**; consulte la
> obra original para el texto completo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ductos import friccion as fr

## 2.1 Ecuación general de energía

Un balance de energía por unidad de masa entre dos secciones de un conducto lleva al gradiente de
presión total como suma de tres términos:

$$\left(\frac{\Delta p}{\Delta L}\right)_T = \left(\frac{\Delta p}{\Delta L}\right)_e + \left(\frac{\Delta p}{\Delta L}\right)_{ac} + \left(\frac{\Delta p}{\Delta L}\right)_f$$

elevación, aceleración y fricción. En los capítulos 3 y 4 el término de aceleración se desprecia.

## 2.2 Darcy y Fanning

$$\left(\frac{\Delta p}{\Delta L}\right)_f = \frac{f\,\rho v^2}{2 g_c d}$$

Fanning usa el radio hidráulico ($R_h = d/4$), por lo que su factor es 4 veces menor.
**Todo el paquete usa el factor de Darcy.**

## 2.2.3 Factor de fricción por régimen

| Régimen | Intervalo | Ecuación |
|---|---|---|
| Laminar | $N_{Re} < 2300$ | $f = 64/N_{Re}$ (2.16) |
| Crítico | $2300 \le N_{Re} \le 3100$ | interpolación (2.19) |
| Turbulento | $N_{Re} > 3100$ | Colebrook-White (2.17), iterativa |
| Francamente turbulento | rugosidad dominante | $f = [-2\log(\varepsilon/3.715d)]^{-2}$ (2.18) |
| Aproximación explícita | $N_{Re} > 3100$ | tipo Jain (2.20) |

### Diagrama de Moody generado con el código

In [ ]:
re = np.logspace(np.log10(600), 8, 400)
fig, ax = plt.subplots(figsize=(9, 6))
for rr in [1e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]:
    f = [fr.factor_friccion(x, rr) for x in re]
    ax.loglog(re, f, lw=1.2, label=f"ε/d = {rr:g}")
ax.axvspan(2300, 3100, color="0.85", label="zona crítica")
ax.set(xlabel="Número de Reynolds", ylabel="f (Darcy)", title="Diagrama de Moody (Ecs. 2.16-2.19)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.show()

### Colebrook iterativo vs. aproximación explícita (Ec. 2.20)

El método de sustituciones sucesivas parte de un valor supuesto de $f$ y converge en pocas iteraciones.
La Ec. 2.20 evita iterar con un error pequeño.

In [ ]:
import pandas as pd
filas = []
for nre in [5e3, 1e4, 1e5, 1e6, 1e7]:
    for rr in [1e-5, 1e-4, 1e-3]:
        fc = fr.f_colebrook(nre, rr)
        fj = fr.f_jain(nre, rr)
        filas.append({"NRe": f"{nre:.0e}", "ε/d": rr, "Colebrook": round(fc, 5),
                      "Ec. 2.20": round(fj, 5), "error %": round(100 * (fj - fc) / fc, 2)})
pd.DataFrame(filas)

### Observación para clase: continuidad en la zona crítica

La Ec. 2.19 tal como aparece en la fuente no empata con $64/N_{Re}$ en $N_{Re}=2300$: da 0.032 frente a 0.0278.
Es un buen punto de discusión sobre el carácter empírico de la transición.

In [ ]:
for nre in [2299, 2300, 2700, 3100, 3101]:
    print(f"NRe={nre:5d}  f={fr.factor_friccion(nre, 0.0006/4):.5f}  ({fr.regimen(nre)})")